# UNSW-NB15 preprocessing (Zeek-compatible features)

Input: the official `UNSW_NB15_training-set.csv`. Output: a stratified **train / validation** split (no test set), a fitted scaler and metadata.

**Model inputs (20 columns, about 39 after one-hot):**
- Direct from Zeek: `dur`, `proto`, `service`, `state`, `sbytes`, `dbytes`, `spkts`, `dpkts`
- Derived: `sload`, `dload`, `smean`, `dmean` (recomputed here), plus `is_sm_ips_ports` and 7 `ct_*` features taken from the file

**How to run:** put this notebook and the CSV in the same folder, set `CSV_PATH` in the config cell, then choose Run All.

## 1. Install packages (first time only)

In [13]:
# Uncomment and run once if a package is missing
# %pip install pandas numpy scikit-learn joblib

## 2. Imports

In [14]:
import json
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight

## 3. Config (set your file path here)

In [15]:
CSV_PATH = "UNSW_NB15_training-set.csv"   # or a full path, e.g. r"C:\Users\YourName\Downloads\UNSW_NB15_training-set.csv"
OUT_DIR = "processed"

RANDOM_STATE = 42
VAL_SIZE = 0.20             # 80% train / 20% validation
DROP_DUPLICATES = False     # keep duplicates: repeated flows are normal in floods (DoS, Generic)
GROUP_DUPLICATES = True     # identical rows always go to the SAME split, so no train/validation leakage
USE_STATE = True            # set False if you decide the state mapping is too shaky
RECOMPUTE_DERIVED = True    # recompute sload/dload/smean/dmean so live == training

## 4. Schema and category groupings (reuse these maps in the live pipeline)

In [16]:
REQUIRED_COLUMNS = [
    "dur", "proto", "service", "state", "sbytes", "dbytes", "spkts", "dpkts",
    "sload", "dload", "smean", "dmean", "is_sm_ips_ports",
    "ct_srv_src", "ct_srv_dst", "ct_dst_ltm", "ct_src_ltm",
    "ct_src_dport_ltm", "ct_dst_sport_ltm", "ct_dst_src_ltm",
    "attack_cat", "label",
]
CT_COLS = ["ct_srv_src", "ct_srv_dst", "ct_dst_ltm", "ct_src_ltm",
           "ct_src_dport_ltm", "ct_dst_sport_ltm", "ct_dst_src_ltm"]
NUMERIC_COLS = ["dur", "sbytes", "dbytes", "spkts", "dpkts",
                "sload", "dload", "smean", "dmean", "is_sm_ips_ports", *CT_COLS]
LOG_COLS = ["dur", "sbytes", "dbytes", "spkts", "dpkts", "sload", "dload", "smean", "dmean"]
NO_SCALE = ["is_sm_ips_ports"]

PROTO_CATS = ["tcp", "udp", "icmp", "other"]
SERVICE_CATS = ["-", "dns", "http", "smtp", "ftp", "ftp-data", "ssh",
                "pop3", "ssl", "irc", "dhcp", "snmp", "radius", "other"]
STATE_GROUPS = ["ok", "no_reply", "reset", "other"]
ARGUS_STATE_MAP = {"FIN": "ok", "CLO": "ok", "CON": "ok", "ACC": "ok",
                   "INT": "no_reply", "REQ": "no_reply", "RST": "reset"}
ZEEK_STATE_MAP = {"SF": "ok", "S1": "ok", "S2": "ok", "S3": "ok", "S0": "no_reply",
                  "REJ": "reset", "RSTO": "reset", "RSTR": "reset",
                  "RSTOS0": "reset", "RSTRH": "reset"}
CAT_SPEC = {"proto": PROTO_CATS, "service": SERVICE_CATS, "state": STATE_GROUPS}

CLASS_NAMES = ["Normal", "DoS", "Reconnaissance", "Fuzzers", "Exploits", "Generic", "Other"]
KEEP_ATTACKS = ["DoS", "Reconnaissance", "Fuzzers", "Exploits", "Generic"]  # the rest become "Other"

## 5. Helper functions

In [17]:
def load_training_csv(path):
    df = pd.read_csv(path, low_memory=False, encoding="latin-1")
    df.columns = [c.strip().lower() for c in df.columns]
    missing = [c for c in REQUIRED_COLUMNS if c not in df.columns]
    if missing:
        raise ValueError(f"Missing expected columns: {missing}\nColumns found: {list(df.columns)}")
    print(f"loaded {len(df):,} rows, {df.shape[1]} columns")
    return df


def group_proto(s):
    s = s.fillna("").astype(str).str.strip().str.lower()
    return s.where(s.isin(PROTO_CATS[:-1]), "other")


def group_service(s):
    s = s.fillna("-").astype(str).str.strip().str.lower().replace("", "-")
    return s.where(s.isin(SERVICE_CATS[:-1]), "other")


def group_state(s, state_map):
    s = s.fillna("").astype(str).str.strip().str.upper()
    return s.map(state_map).fillna("other")


def safe_div(a, b):
    a = np.asarray(a, dtype="float64")
    b = np.asarray(b, dtype="float64")
    out = np.zeros_like(a)
    np.divide(a, b, out=out, where=b > 0)
    return out


def clean_and_engineer(df, state_map, recompute=RECOMPUTE_DERIVED):
    df = df.copy()
    for c in NUMERIC_COLS:
        df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0)
    df["label"] = pd.to_numeric(df["label"], errors="coerce").fillna(0).astype(int)

    df["proto"] = group_proto(df["proto"])
    df["service"] = group_service(df["service"])
    df["state"] = group_state(df["state"], state_map)

    if recompute:
        df["sload"] = safe_div(df["sbytes"] * 8, df["dur"])
        df["dload"] = safe_div(df["dbytes"] * 8, df["dur"])
        df["smean"] = safe_div(df["sbytes"], df["spkts"])
        df["dmean"] = safe_div(df["dbytes"], df["dpkts"])

    df["is_sm_ips_ports"] = (df["is_sm_ips_ports"] > 0).astype(int)
    for c in CT_COLS:
        df[c] = df[c].clip(lower=0)
    return df


def build_targets(df):
    cat = (df["attack_cat"].fillna("Normal").astype(str).str.strip()
           .replace({"Backdoor": "Backdoors", "": "Normal"}))
    group = cat.where(cat.isin(["Normal", *KEEP_ATTACKS]), "Other")
    group = group.mask(df["label"] == 0, "Normal")
    group = group.mask((df["label"] == 1) & (group == "Normal"), "Other")
    df = df.copy()
    df["attack_group"] = group
    df["y_multi"] = group.map({c: i for i, c in enumerate(CLASS_NAMES)}).astype(int)
    return df


def build_model_frame(df, use_state=USE_STATE):
    num = df[NUMERIC_COLS].astype("float64").copy()
    for c in LOG_COLS:
        num[c] = np.log1p(num[c].clip(lower=0))
    parts = [num]
    for col, cats in CAT_SPEC.items():
        if col == "state" and not use_state:
            continue
        cat = pd.Series(pd.Categorical(df[col], categories=cats), index=df.index)
        parts.append(pd.get_dummies(cat, prefix=col).astype("float64"))
    return pd.concat(parts, axis=1)


def scale_columns(frame):
    return [c for c in frame.columns if c in NUMERIC_COLS and c not in NO_SCALE]

## 6. Load the data

In [18]:
raw = load_training_csv(CSV_PATH)
raw.head()

loaded 175,341 rows, 45 columns


,id,dur,proto,service,state,spkts,dpkts,sbytes,dbytes,rate,...,ct_dst_sport_ltm,ct_dst_src_ltm,is_ftp_login,ct_ftp_cmd,ct_flw_http_mthd,ct_src_ltm,ct_srv_dst,is_sm_ips_ports,attack_cat,label
0,1,0.121478,tcp,-,FIN,6,4,258,172,74.087490,...,1,1,0,0,0,1,1,0,Normal,0
1,2,0.649902,tcp,-,FIN,14,38,734,42014,78.473372,...,1,2,0,0,0,1,6,0,Normal,0
2,3,1.623129,tcp,-,FIN,8,16,364,13186,14.170161,...,1,3,0,0,0,2,6,0,Normal,0
3,4,1.681642,tcp,ftp,FIN,12,12,628,770,13.677108,...,1,3,1,1,0,2,1,0,Normal,0
4,5,0.449454,tcp,-,FIN,10,6,534,268,33.373826,...,1,40,0,0,0,2,39,0,Normal,0


## 7. Clean, engineer features and build targets

In [19]:
df = clean_and_engineer(raw, ARGUS_STATE_MAP)
df = build_targets(df)
frame = build_model_frame(df)
y_bin = df["label"].to_numpy()
y_multi = df["y_multi"].to_numpy()

print("feature columns:", frame.shape[1])
print(df["attack_group"].value_counts())

feature columns: 39
attack_group
Normal            56000
Generic           40000
Exploits          33393
Fuzzers           18184
DoS               12264
Reconnaissance    10491
Other              5009
Name: count, dtype: int64


## 8. Remove duplicates (off by default, see config)

In [20]:
if DROP_DUPLICATES:
    key = frame.copy()
    key["__y"] = y_multi
    keep = ~key.duplicated().to_numpy()
    print(f"dropping {(~keep).sum():,} duplicate rows")
    frame = frame[keep].reset_index(drop=True)
    y_bin, y_multi = y_bin[keep], y_multi[keep]
print("rows kept:", len(frame))

rows kept: 175341


## 9. Stratified train / validation split

With `GROUP_DUPLICATES = True`, rows that are identical (same features and same class) are kept together in one split, so the validation set never contains an exact copy of a training row. Classes stay stratified, and the natural class balance is kept.

In [21]:
from sklearn.model_selection import StratifiedGroupKFold

idx = np.arange(len(frame))
if GROUP_DUPLICATES:
    key_df = frame.copy()
    key_df["__y"] = y_multi
    groups = key_df.groupby(list(key_df.columns), sort=False).ngroup().to_numpy()
    print(f"{len(frame):,} rows form {groups.max() + 1:,} distinct groups")
    sgkf = StratifiedGroupKFold(n_splits=round(1 / VAL_SIZE), shuffle=True, random_state=RANDOM_STATE)
    idx_train, idx_val = next(sgkf.split(frame, y_multi, groups))
else:
    idx_train, idx_val = train_test_split(
        idx, test_size=VAL_SIZE, stratify=y_multi, random_state=RANDOM_STATE)

print(f"train: {len(idx_train):,}   validation: {len(idx_val):,}  ({100 * len(idx_val) / len(idx):.1f}% validation)")

share = pd.DataFrame({
    "all": pd.Series(y_multi).value_counts(normalize=True),
    "train": pd.Series(y_multi[idx_train]).value_counts(normalize=True),
    "val": pd.Series(y_multi[idx_val]).value_counts(normalize=True),
}).sort_index()
share.index = [CLASS_NAMES[i] for i in share.index]
(share * 100).round(2)

175,341 rows form 104,608 distinct groups
train: 140,272   validation: 35,069  (20.0% validation)


,all,train,val
Normal,31.94,31.94,31.94
DoS,6.99,6.99,6.99
Reconnaissance,5.98,5.98,5.99
Fuzzers,10.37,10.37,10.37
Exploits,19.04,19.05,19.04
Generic,22.81,22.81,22.81
Other,2.86,2.86,2.86


## 10. Scale (scaler is fit on the train set only)

In [22]:
cols = scale_columns(frame)
scaler = StandardScaler().fit(frame.iloc[idx_train][cols])
scaled = frame.copy()
scaled[cols] = scaler.transform(frame[cols])
X = scaled.to_numpy(dtype="float32")

splits = {"train": idx_train, "val": idx_val}
arrays = {}
for name, ix in splits.items():
    arrays[f"X_{name}"] = X[ix]
    arrays[f"y_bin_{name}"] = y_bin[ix].astype("int8")
    arrays[f"y_multi_{name}"] = y_multi[ix].astype("int8")

## 11. Save arrays, scaler and metadata

In [23]:
out = Path(OUT_DIR)
out.mkdir(parents=True, exist_ok=True)

np.savez_compressed(out / "unsw_nb15_processed.npz", **arrays)
joblib.dump(scaler, out / "scaler.joblib")

cw_multi = compute_class_weight("balanced", classes=np.arange(len(CLASS_NAMES)), y=y_multi[idx_train])
cw_bin = compute_class_weight("balanced", classes=np.array([0, 1]), y=y_bin[idx_train])
meta = {
    "feature_names": list(frame.columns),
    "scale_cols": cols,
    "class_names": CLASS_NAMES,
    "class_weight_multi": {int(i): float(w) for i, w in enumerate(cw_multi)},
    "class_weight_binary": {0: float(cw_bin[0]), 1: float(cw_bin[1])},
    "config": {"use_state": USE_STATE, "recompute_derived": RECOMPUTE_DERIVED,
               "drop_duplicates": DROP_DUPLICATES, "group_duplicates": GROUP_DUPLICATES,
               "random_state": RANDOM_STATE, "val_size": VAL_SIZE},
    "zeek_state_map": ZEEK_STATE_MAP,
}
(out / "meta.json").write_text(json.dumps(meta, indent=2))
print("saved to", out.resolve())

saved to D:\College\THAPAR\SEM 5\Machine Learning\Project\processed


## 11b. Save separate train and validation files

Writes `train.csv` and `validation.csv` (scaled features plus the two targets and the attack class name), so you can open them in Excel or pandas. The `.npz` file above still holds the same data for the model notebooks.

In [24]:
class_lookup = dict(enumerate(CLASS_NAMES))
for name, ix in splits.items():
    part = pd.DataFrame(X[ix], columns=frame.columns)
    part["label"] = y_bin[ix]
    part["attack_class_id"] = y_multi[ix]
    part["attack_class"] = pd.Series(y_multi[ix]).map(class_lookup).to_numpy()
    fname = "train.csv" if name == "train" else "validation.csv"
    part.to_csv(out / fname, index=False)
    print(f"{fname:16s} {part.shape[0]:>9,} rows x {part.shape[1]} columns")

train.csv          140,272 rows x 42 columns
validation.csv      35,069 rows x 42 columns


## 12. Summary of the split

In [25]:
for name, ix in splits.items():
    counts = pd.Series(y_multi[ix]).map(dict(enumerate(CLASS_NAMES))).value_counts()
    print(f"\n{name}: {len(ix):,} rows, {100 * y_bin[ix].mean():.1f}% attack")
    print(counts.to_string())


train: 140,272 rows, 68.1% attack
Normal            44800
Generic           32000
Exploits          26715
Fuzzers           14547
DoS                9811
Reconnaissance     8392
Other              4007

val: 35,069 rows, 68.1% attack
Normal            11200
Generic            8000
Exploits           6678
Fuzzers            3637
DoS                2453
Reconnaissance     2099
Other              1002


## 13. Live helper (use later in the Zeek pipeline)

Takes an unscaled model frame built with `build_model_frame()`, aligns it to the training columns and applies the saved scaler.

In [26]:
def transform_with_artifacts(frame, artifacts_dir=OUT_DIR):
    art = Path(artifacts_dir)
    scaler = joblib.load(art / "scaler.joblib")
    meta = json.loads((art / "meta.json").read_text())
    frame = frame.reindex(columns=meta["feature_names"], fill_value=0.0).copy()
    frame[meta["scale_cols"]] = scaler.transform(frame[meta["scale_cols"]])
    return frame.to_numpy(dtype="float32")